In [62]:
import bw2data as bd
import bw2io as bi
from bw2io import BW2Package
from bw2calc import LCA
from pathlib import Path
import bw2analyzer as ba
import pandas as pd
from IPython.display import display


In [63]:
PROJECT_NAME = "Lasers4MaaS"
ECOINVENT_DATABASE_NAME = "ecoinvent-3.12-cutoff"

bd.projects.set_current(PROJECT_NAME)
if ECOINVENT_DATABASE_NAME not in bd.databases:
    bi.import_ecoinvent_release(version="3.12", system_model="cutoff")
else:
    print(f"Database {ECOINVENT_DATABASE_NAME} already exists, skipping import.")

Database ecoinvent-3.12-cutoff already exists, skipping import.


In [64]:
ei = bd.Database(ECOINVENT_DATABASE_NAME)

### Variables: bio


In [65]:
bio = bd.Database("ecoinvent-3.12-biosphere")

## L4M R&D: parameter sets per use case
**Goal:** identify a set of optimised welding parameters for each use case (Fusion welding, Pharma welding, Battery cooling welding).
**Reference flow:** 1 set of optimised parameters for one use case.
**Scenarios per use case:**
- *SoA R&D*: all experiments run physically in the lab with the standard laser.
- *L4M-enabled R&D*: fewer physical experiments, plus digital lab runs, with the joining done by Dynamic Beam Shaping (DBS).

Data come from `databases/L4M_database.bw2package` (`L4M system`).

In [66]:
import re
import hashlib
import pickle
import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PACKAGE_DIR = Path("databases")
L4M_NAME = "L4M system"
METALLURGIC_NAME = "Mettalurgic test (CIVAN)"      # spelling as stored in the packages
CT_SCAN_NAME = "CT-scan test"
FOREGROUND_DBS = [L4M_NAME, METALLURGIC_NAME, CT_SCAN_NAME]

# Set to True to delete and re-import L4M system from the package (discards edits made in the project)
REIMPORT_L4M = False

def ensure_package(db_name, package_file, force=False):
    """Import a .bw2package unless the database already holds activities."""
    if db_name in bd.databases and len(bd.Database(db_name)) > 0 and not force:
        print(f"{db_name}: {len(bd.Database(db_name))} activities already in the project, skipping import.")
        return
    if db_name in bd.databases:
        del bd.databases[db_name]
    BW2Package.import_file(str(PACKAGE_DIR / package_file))
    print(f"{db_name}: imported {len(bd.Database(db_name))} activities from {package_file}")

# L4M system links to the other two databases, so they come first
ensure_package(CT_SCAN_NAME, "CT-scan test.bw2package")
ensure_package(METALLURGIC_NAME, "Metallurgic test (CIVAN).bw2package")
ensure_package(L4M_NAME, "L4M_database.bw2package", force=REIMPORT_L4M)

for db in FOREGROUND_DBS:
    for a in bd.Database(db):
        for e in a.exchanges():          # raises UnknownObject for a dangling link
            e.input
    bd.Database(db).process()            # rebuild the matrices so freshly imported dependencies are picked up
l4m = bd.Database(L4M_NAME)

CT-scan test: 5 activities already in the project, skipping import.
Mettalurgic test (CIVAN): 9 activities already in the project, skipping import.
L4M system: 11 activities already in the project, skipping import.


### Impact indicators (same EF 3.1 selection as `joining_task.ipynb`)

In [67]:
most_used_indicators = [
    'climate change: total',
    'particulate matter formation',
    'acidification',
    'eutrophication: terrestrial',
    'photochemical oxidant formation: human health',
    'human toxicity: carcinogenic'
]
ef31_methods = [
    m for m in bd.methods
    if ('EF v3.1' in str(m) or 'EF 3.1' in str(m))
    and 'no LT' not in str(m)
    and any(indicator in str(m) for indicator in most_used_indicators)
]
method_units = {m[2]: bd.Method(m).metadata.get("unit", "") for m in ef31_methods}
indicators = [m[2] for m in ef31_methods]
DEFAULT_INDICATOR = next((i for i in indicators if i.startswith("climate change")), indicators[0])
short = lambda s: s.split(" (EF")[0]          # drop the "(EF v3.1 - IPCC 2021)" suffix in titles

### Use cases and R&D scenarios
- One physical experiment = one coupon weld (`001_Physical_Laser welding`, which pulls in the 2 cleaned and cut coupons it needs through the Metallurgic cleaning and cutting steps) plus the use-case welding process. `001_Physical lab` is not used: it demands cutting, cleaning and welding in parallel, so the coupons would be counted twice. In *L4M-enabled R&D* every joining step (coupon joining, prototype joining and the weld inside the CT-scan sample) uses DBS, which consumes 20% less electricity than the standard laser (`DBS_ENERGY_FACTOR`). Argon is unchanged.
- One digital run = `Digital lab processes` (platform + prediction).
- Per set of parameters, the optimised welds are verified once by a metallographic sample (`08_Etching`, post-processing: cross-section, mounting, grinding, polishing, etching) and one CT-scan (`CT-scan X-ray`, testing). Both pull in the weld they inspect, as modelled in the package. Only the weld itself is changed by DBS; the cutting around it is not.

**The experiment counts below are placeholders, not project data. Replace them with the agreed numbers.**

In [68]:
def l4m_process(name):
    return next(a for a in l4m if a["name"] == name or a["name"].startswith(name))

USE_CASES = {                       # use case -> its welding process in L4M system
    "Fusion welding":        l4m_process("Fusion welding"),
    "Pharma welding":        l4m_process("Pharma welding"),
    "Battery cooling welding": l4m_process("Battery cooling welding"),
}
digital_lab  = l4m_process("Digital lab processes")
ct_scan      = l4m_process("CT-scan X-ray")
metallography = next(a for a in bd.Database(METALLURGIC_NAME) if a["name"] == "08_Etching")

# Dynamic Beam Shaping consumes 20% less energy than the standard laser (applied to all electricity of the joining steps)
DBS_ENERGY_FACTOR = 0.8
coupon_joining = l4m_process("001_Physical_Laser welding")

# PLACEHOLDERS: experiments (and verification samples) needed to identify 1 set of optimised parameters;
# "joining" is the technology used for the coupon and prototype joining
RD_SCENARIOS = {
    "Fusion welding": {
        "SoA R&D":              {"joining": "Standard laser", "physical": 500, "digital": 0, "ct_scan": 1, "metallography": 1},
        "L4M-enabled R&D":      {"joining": "Dynamic Beam Shaping", "physical": 275,  "digital": 1, "ct_scan": 1, "metallography": 1},
    },
    "Pharma welding": {
        "SoA R&D":              {"joining": "Standard laser", "physical": 500, "digital": 0, "ct_scan": 1, "metallography": 1},
        "L4M-enabled R&D":      {"joining": "Dynamic Beam Shaping", "physical":275,  "digital": 1, "ct_scan": 1, "metallography": 1},
    },
    "Battery cooling welding": {
        "SoA R&D":              {"joining": "Standard laser", "physical": 500, "digital": 0, "ct_scan": 1, "metallography": 1},
        "L4M-enabled R&D":      {"joining": "Dynamic Beam Shaping", "physical": 275,  "digital": 1, "ct_scan": 1, "metallography": 1},
    },
}

# One product system per use case and scenario; reference flow = 1 set of optimised parameters
RD_NAME = "L4M_RD_parameter_sets"
rd_data = {}

def dbs_copy(proc, replace=None, reduce=True):
    """Copy of a process (name unchanged, code suffixed _DBS). With `reduce`, its electricity is cut by DBS_ENERGY_FACTOR
    (joining steps); `replace` maps input keys to other keys so parents can be pointed at the DBS copies."""
    key = (RD_NAME, proc["code"] + "_DBS")
    exchanges = [{"input": key, "amount": next(e["amount"] for e in proc.exchanges() if e["type"] == "production"), "type": "production"}]
    for e in proc.exchanges():
        if e["type"] == "production":
            continue
        amount = e["amount"] * (DBS_ENERGY_FACTOR if reduce and "electricity" in e.input["name"] else 1)
        exchanges.append({"input": (replace or {}).get(e["input"], e["input"]), "amount": amount, "type": e["type"],
                          **({"comment": e["comment"]} if e.get("comment") else {})})
    rd_data[key] = {"name": proc["name"], "reference product": proc.get("reference product"), "unit": proc["unit"],
                    "location": proc["location"], "technology": "Dynamic Beam Shaping", "exchanges": exchanges}
    return key

# DBS versions of the coupon joining and of each use-case welding process
dbs_coupon = dbs_copy(coupon_joining)
dbs_welding = {uc: dbs_copy(p) for uc, p in USE_CASES.items()}

# The CT-scan sample contains its own weld: CT-scan X-ray -> 04_Cutting -> 03_Laser welding (CT-scan test database)
ct_cut  = next(e.input for e in ct_scan.exchanges() if e["type"] == "technosphere" and e.input["database"] == CT_SCAN_NAME)
ct_weld = next(e.input for e in ct_cut.exchanges() if e["type"] == "technosphere" and e.input["database"] == CT_SCAN_NAME and "Laser welding" in e.input["name"])
dbs_ct_weld = dbs_copy(ct_weld)
dbs_ct_cut  = dbs_copy(ct_cut,  replace={ct_weld.key: dbs_ct_weld}, reduce=False)
dbs_ct_scan = dbs_copy(ct_scan, replace={ct_cut.key: dbs_ct_cut},   reduce=False)

for uc, scenarios in RD_SCENARIOS.items():
    for sc, n in scenarios.items():
        key = (RD_NAME, f"{uc} | {sc}")
        exchanges = [{"input": key, "amount": 1, "type": "production"}]
        dbs = n["joining"] == "Dynamic Beam Shaping"
        if n["physical"]:
            exchanges += [{"input": dbs_coupon if dbs else coupon_joining.key, "amount": n["physical"], "type": "technosphere"},
                          {"input": dbs_welding[uc] if dbs else USE_CASES[uc].key, "amount": n["physical"], "type": "technosphere"}]
        for count, key_ in ((n["digital"], digital_lab.key), (n["ct_scan"], dbs_ct_scan if dbs else ct_scan.key), (n["metallography"], metallography.key)):
            if count:
                exchanges.append({"input": key_, "amount": count, "type": "technosphere"})
        rd_data[key] = {"name": f"{uc} | {sc}", "reference product": "1 set of optimised parameters",
                        "unit": "set of parameters", "location": "GLO", "exchanges": exchanges}
rd = bd.Database(RD_NAME)
rd.write(rd_data)

systems = {(uc, sc): rd.get(f"{uc} | {sc}") for uc in RD_SCENARIOS for sc in RD_SCENARIOS[uc]}
pd.DataFrame([{"Use case": uc, "Scenario": sc, **n} for uc, s in RD_SCENARIOS.items() for sc, n in s.items()])

100%|██████████| 13/13 [00:00<00:00, 14398.19it/s]

13:12:34+0100 [info     ] Vacuuming database            


,Use case,Scenario,joining,physical,digital,ct_scan,metallography
0,Fusion welding,SoA R&D,Standard laser,500,0,1,1
1,Fusion welding,L4M-enabled R&D,Dynamic Beam Shaping,275,1,1,1
2,Pharma welding,SoA R&D,Standard laser,500,0,1,1
3,Pharma welding,L4M-enabled R&D,Dynamic Beam Shaping,275,1,1,1
4,Battery cooling welding,SoA R&D,Standard laser,500,0,1,1
5,Battery cooling welding,L4M-enabled R&D,Dynamic Beam Shaping,275,1,1,1


### Impact calculation
Each product system is walked down through the foreground processes to the background inputs (ecoinvent markets, biosphere flows). Every background input is characterised once per indicator; the walk scales it by the amount needed for 1 set of parameters. The total is checked against a regular `bw2calc` LCA, and the same walk feeds the stacked bar, Sankey and regional views.

In [69]:
M = len(ef31_methods)

def _unit_impacts(act):
    lca = LCA({act: 1}, method=ef31_methods[0]); lca.lci()
    out = []
    for m in ef31_methods:
        lca.switch_method(m); lca.lcia(); out.append(lca.score)
    return np.array(out)

_cf = [{row[0]: row[1] for row in bd.Method(m).load()} for m in ef31_methods]
_q = {}
def background_impact(key, kind):
    if (key, kind) not in _q:
        _q[key, kind] = np.array([cf.get(key, 0.0) for cf in _cf]) if kind == "biosphere" else _unit_impacts(bd.get_activity(key))
    return _q[key, kind]

is_electricity = lambda key: bool(re.match(r"market (group )?for electricity, medium voltage", bd.get_activity(key)["name"]))

def trace(act, mult=1.0, path=()):
    """Walk the foreground below `act` (demanded `mult` times); yield (path of processes, background key, kind, amount)."""
    path = path + (act.key,)
    prod = next(e["amount"] for e in act.exchanges() if e["type"] == "production")
    for e in act.exchanges():
        if e["type"] == "production":
            continue
        amount = mult * e["amount"] / prod
        if e["input"][0] in [*FOREGROUND_DBS, RD_NAME]:
            yield from trace(bd.get_activity(e["input"]), amount, path)
        else:
            yield path, e["input"], e["type"], amount

records = {k: list(trace(act)) for k, act in systems.items()}

# In a DBS scenario no standard-laser weld may remain anywhere in the walk
for (uc, sc), recs in records.items():
    if RD_SCENARIOS[uc][sc]["joining"] == "Dynamic Beam Shaping":
        std = {bd.get_activity(p)["name"] for path, *_ in recs for p in path
               if "Laser welding" in bd.get_activity(p)["name"] and bd.get_activity(p)["database"] != RD_NAME}
        assert not std, f"Standard-laser weld left in {uc} | {sc}: {std}"

def contributions(k, country=None):
    """One row per background flow: path, key, impact vector. `country` swaps all electricity for that market."""
    rows = []
    for path, key, kind, amount in records[k]:
        q = grid[country] if country and kind == "technosphere" and is_electricity(key) else background_impact(key, kind)
        rows.append((path, key, kind, amount * q))
    return rows

In [70]:
# Electricity market of each country, for the regional analysis (EU27 plus GB, which the physical lab is modelled with)
EU27 = {"AT":"AUT","BE":"BEL","BG":"BGR","HR":"HRV","CY":"CYP","CZ":"CZE","DK":"DNK","EE":"EST","FI":"FIN",
        "FR":"FRA","DE":"DEU","GR":"GRC","HU":"HUN","IE":"IRL","IT":"ITA","LV":"LVA","LT":"LTU","LU":"LUX",
        "MT":"MLT","NL":"NLD","PL":"POL","PT":"PRT","RO":"ROU","SK":"SVK","SI":"SVN","ES":"ESP","SE":"SWE"}
COUNTRIES = {**EU27, "GB": "GBR"}

FORCE_RECALC = False
CACHE_DIR = Path("results/.cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
_key = hashlib.sha256(repr((ECOINVENT_DATABASE_NAME, sorted(COUNTRIES), list(ef31_methods))).encode()).hexdigest()[:16]
_cache = CACHE_DIR / f"l4m_grid_{_key}.pkl"
if _cache.exists() and not FORCE_RECALC:
    with open(_cache, "rb") as f:
        grid = pickle.load(f)
    print(f"Loaded cached electricity-market impacts from {_cache} (set FORCE_RECALC = True to recompute)")
else:
    grid = {}
    for iso2 in COUNTRIES:
        market = next(a for a in ei if a["name"] == "market for electricity, medium voltage" and a["location"] == iso2)
        grid[iso2] = _unit_impacts(market)
    with open(_cache, "wb") as f:
        pickle.dump(grid, f)
    print(f"Computed electricity-market impacts and cached them in {_cache}")

# Totals per product system, checked against bw2calc
totals = {k: sum(c[3] for c in contributions(k)) for k in systems}
for k, act in systems.items():
    ref = _unit_impacts(act)
    assert np.allclose(totals[k], ref, rtol=1e-6), f"Walk does not reproduce bw2calc for {k}"

results = pd.DataFrame({k: dict(zip(indicators, v)) for k, v in totals.items()}).T.rename_axis(["Use case", "Scenario"])
results[[DEFAULT_INDICATOR]].unstack("Scenario").droplevel(0, axis=1)

Loaded cached electricity-market impacts from results/.cache/l4m_grid_3a2c85dffa83c1c5.pkl (set FORCE_RECALC = True to recompute)


/opt/anaconda3/envs/brightway/lib/python3.11/site-packages/scikits/umfpack/umfpack.py:737: UmfpackWarning: (almost) singular matrix! (estimated cond. number: 7.65e+12)
  warnings.warn(msg, UmfpackWarning)


Scenario,L4M-enabled R&D,SoA R&D
Use case,,
Battery cooling welding,103.679157,178.996776
Fusion welding,137.832561,252.685624
Pharma welding,115.832341,204.848013


## Stacked bar: SoA R&D vs L4M-enabled R&D
Impact of 1 set of optimised parameters, split by the process that causes it. Processes are grouped into pre-processing, joining, post-processing, testing and digital lab; the legend follows the same grouping.

In [75]:
# Process types and their processes (legend order); one colour family per type
PROCESS_TYPES = {
    "Pre-processing":  {"Cutting": "#2a78d6", "Cleaning": "#8fb8ee"},
    "Joining":         {"Coupon joining": "#eb6834", "Prototype joining": "#f4a582"},
    "Post-processing": {"Cutting (weld cross-sections)": "#4b3bb5", "Hot-mounting": "#6c5cd0", "Grinding": "#8d80e0",
                        "Polishing": "#b0a6ec", "Etching": "#d3cdf6"},
    "Testing":         {"CT-scan X-ray": "#1baf7a"},
    "Digital lab":     {"Platform": "#eda100", "Prediction": "#e87ba4"},
}
use_case_welding_names = {p["name"] for p in USE_CASES.values()}

def process_label(key):
    name = bd.get_activity(key)["name"]
    if name in use_case_welding_names:
        return "Prototype joining"
    if name.startswith("04_Cutting"):
        return "Cutting (weld cross-sections)"
    label = re.sub(r"^\d+_(Physical_|Pyshical_)?", "", name)
    return "Coupon joining" if label == "Laser welding" else label

def stack_table(ind):
    """Impact per set of parameters caused directly by each process (rows: systems, columns: process labels)."""
    k = indicators.index(ind)
    out = {}
    for sysk in systems:
        s = {}
        for path, key, kind, v in contributions(sysk):
            lab = process_label(path[-1])
            s[lab] = s.get(lab, 0.0) + v[k]
        out[sysk] = s
    return pd.DataFrame(out).T.fillna(0.0)

w_ind = widgets.Dropdown(options=indicators, value=DEFAULT_INDICATOR, description="Indicator", layout=widgets.Layout(width="520px"), style={"description_width": "80px"})
fig_bar = go.FigureWidget()

def draw_bars(_=None):
    d = stack_table(w_ind.value)
    unknown = [c for c in d.columns if not any(c in procs for procs in PROCESS_TYPES.values())]
    assert not unknown, f"Processes without a type: {unknown}"
    unit = method_units.get(w_ind.value, "")
    x = [[uc for uc, _ in d.index], [sc for _, sc in d.index]]
    tot = d.sum(axis=1)
    with fig_bar.batch_update():
        fig_bar.data = []
        for ptype, procs in PROCESS_TYPES.items():
            first = True
            for proc, color in procs.items():
                if proc not in d.columns:
                    continue
                fig_bar.add_bar(x=x, y=d[proc], name=proc, legendgroup=ptype, legendgrouptitle_text=f"<b>{ptype}</b>" if first else None,
                                marker=dict(color=color, line=dict(color="white", width=1)),
                                customdata=(d[proc] / tot * 100),
                                hovertemplate=f"<b>%{{x}}</b><br>{ptype}: {proc}<br>%{{y:.3g}} {unit} (%{{customdata:.1f}}% of total)<extra></extra>")
                first = False
        fig_bar.add_scatter(x=x, y=tot, mode="text", text=[f"{t:.3g}" for t in tot], textposition="top center", showlegend=False, hoverinfo="skip")
        fig_bar.update_layout(barmode="relative", template="plotly_white", height=900, width=1260, font=dict(size=14),
                              title=dict(text=f"{short(w_ind.value)} per set of optimised parameters", x=0.01, font=dict(size=15)),
                              yaxis=dict(title=unit, gridcolor="#e6e9ec"), legend=dict(groupclick="toggleitem", tracegroupgap=14),
                              margin=dict(l=70, r=20, t=80, b=50))

w_ind.observe(draw_bars, names="value")
draw_bars()
display(widgets.VBox([w_ind, fig_bar]))

## Sankey: where the impact flows
Background inputs (left) feed the foreground processes, which pass their cumulative impact on to the set of parameters (right).

In [72]:
DB_SHORT = {L4M_NAME: "L4M", METALLURGIC_NAME: "Metallurgic", CT_SCAN_NAME: "CT-scan", RD_NAME: ""}
def node_label(key):
    act = bd.get_activity(key)
    tag = DB_SHORT.get(act["database"], "")
    return f"{act['name']} ({tag})" if tag else act["name"]

def sankey_data(sysk, ind):
    k = indicators.index(ind)
    flows = {}
    for path, key, kind, v in contributions(sysk):
        leaf = f"{bd.get_activity(key)['name'][:40]} [input]" if kind == "technosphere" else f"{bd.get_activity(key)['name']} [flow]"
        nodes_ = [leaf] + [node_label(p) for p in reversed(path)]
        for a, b in zip(nodes_[:-1], nodes_[1:]):
            flows[a, b] = flows.get((a, b), 0.0) + v[k]
    labels = list(dict.fromkeys(n for ab in flows for n in ab))
    pos = [(labels.index(a), labels.index(b), v) for (a, b), v in flows.items() if v > 0]   # negative flows can't be drawn
    return labels, [p[0] for p in pos], [p[1] for p in pos], [p[2] for p in pos]

sk_opts = [f"{uc} | {sc}" for uc, sc in systems]
w_sys  = widgets.Dropdown(options=sk_opts, value=sk_opts[0], description="System", layout=widgets.Layout(width="520px"), style={"description_width": "80px"})
w_sind = widgets.Dropdown(options=indicators, value=DEFAULT_INDICATOR, description="Indicator", layout=widgets.Layout(width="520px"), style={"description_width": "80px"})
fig_sankey = go.FigureWidget(go.Sankey(node=dict(pad=14, thickness=14), link=dict(source=[], target=[], value=[])))

def draw_sankey(_=None):
    sysk = tuple(w_sys.value.split(" | "))
    labels, src, tgt, val = sankey_data(sysk, w_sind.value)
    unit = method_units.get(w_sind.value, "")
    colors = ["#2a78d6" if l.endswith(("[input]", "[flow]")) else ("#eb6834" if l == systems[sysk]["name"] else "#8a8a85") for l in labels]
    total = totals[sysk][indicators.index(w_sind.value)]
    with fig_sankey.batch_update():
        t = fig_sankey.data[0]
        t.node.label, t.node.color = labels, colors
        t.link.source, t.link.target, t.link.value = src, tgt, val
        t.link.color = "rgba(138,138,133,0.3)"
        t.valueformat, t.valuesuffix = ".3g", f" {unit}"
        fig_sankey.update_layout(template="plotly_white", height=620, font=dict(size=11), margin=dict(l=10, r=10, t=60, b=10),
                                 title=dict(text=f"{w_sys.value}: {short(w_sind.value)}, net total {total:.4g} {unit}", x=0.01, font=dict(size=15)))

for w in (w_sys, w_sind): w.observe(draw_sankey, names="value")
draw_sankey()
display(widgets.VBox([w_sys, w_sind, fig_sankey]))

## Regional variation: EU electricity markets
Every electricity input (the RER market group and the GB markets used by the physical lab) is replaced by one country's market. Left: SoA R&D, right: L4M-enabled R&D.

In [73]:
countries = list(COUNTRIES)
iso3 = [COUNTRIES[c] for c in countries]
reg = {}
for uc, sc in systems:
    reg[uc, sc, "As modelled"] = totals[uc, sc]
    for c in countries:
        reg[uc, sc, c] = sum(row[3] for row in contributions((uc, sc), c))

SC_FULL, SC_HYBRID = "SoA R&D", "L4M-enabled R&D"
wide = widgets.Layout(width="420px"); sty = {"description_width": "90px"}
r_ind  = widgets.Dropdown(options=indicators, value=DEFAULT_INDICATOR, description="Indicator", layout=wide, style=sty)
r_uc   = widgets.Dropdown(options=list(RD_SCENARIOS), description="Use case", layout=wide, style=sty)
r_view = widgets.Dropdown(options=["Absolute", "Change vs as modelled (%)"], description="View", layout=wide, style=sty)

fig_reg = go.FigureWidget(make_subplots(rows=1, cols=2, specs=[[{"type": "choropleth"}, {"type": "choropleth"}]],
                                        subplot_titles=(SC_FULL, SC_HYBRID), horizontal_spacing=0.04))
for col, x in ((1, 0.46), (2, 1.0)):
    fig_reg.add_trace(go.Choropleth(locations=iso3, z=[0] * len(countries), text=countries, hovertemplate="<b>%{text}</b><br>%{z:.4g}<extra></extra>",
                                    colorbar=dict(x=x, len=0.8, thickness=12, outlinewidth=0, tickfont=dict(size=10)),
                                    marker=dict(line=dict(color="white", width=0.5))), row=1, col=col)
fig_reg.update_geos(scope="europe", fitbounds="locations", showframe=False, showcountries=True, countrycolor="#c4cad0",
                    showland=True, landcolor="#eceff1", showocean=False, showlakes=False, bgcolor="rgba(0,0,0,0)")
fig_reg.update_layout(template="plotly_white", height=520, font=dict(size=12), margin=dict(l=20, r=20, t=80, b=20))

def update_reg(_=None):
    k = indicators.index(r_ind.value)
    pct = r_view.value != "Absolute"
    zs = []
    for sc in (SC_FULL, SC_HYBRID):
        z = np.array([reg[r_uc.value, sc, c][k] for c in countries])
        base = reg[r_uc.value, sc, "As modelled"][k]
        zs.append((z / base - 1) * 100 if pct else z)
    lim = max(np.abs(np.concatenate(zs)).max(), 1e-12)
    lo, hi = (-lim, lim) if pct else (min(z.min() for z in zs), max(z.max() for z in zs))
    unit = "% vs as modelled" if pct else method_units.get(r_ind.value, "")
    with fig_reg.batch_update():
        for trace_, z in zip(fig_reg.data, zs):          # shared colour scale so both scenarios are comparable
            trace_.z, trace_.zmin, trace_.zmax = z, lo, hi
            trace_.colorscale = "RdBu_r" if pct else "YlOrRd"
            trace_.colorbar.title.text = unit
        fig_reg.layout.title = dict(text=f"{r_uc.value}: {short(r_ind.value)} per set of parameters", x=0.01, font=dict(size=15))

for w in (r_ind, r_uc, r_view): w.observe(update_reg, names="value")
update_reg()
display(widgets.VBox([widgets.HBox([r_ind, r_uc]), r_view, fig_reg]))